# AdMarket Arena — Oversight Agent GRPO Training

Trains the **OversightAgent** (Fleet AI Scalable Oversight bonus) on the AdMarket Arena env via TRL GRPO + Unsloth 4-bit LoRA.

**T4 (Colab) tuned config** — base model: Qwen2.5-3B-Instruct (4-bit NF4 + LoRA r=16), GRPO `num_generations=4`, 256-token output cap, 8192-token context, ~3-4 hr expected run on Colab T4.

Decoupled from advertiser training: the dataset (`data/oversight_train_trajectories.jsonl`) is pre-generated by `scripts/collect_oversight_trajectories.py` using a frozen rule-based pacing advertiser × `violation_injector`, so this notebook can run in parallel with or after the advertiser training notebook.

**Inputs**: `data/oversight_train_trajectories.jsonl` (200 episodes recommended, ~1400 day-records).

**Outputs**:
- `checkpoints/oversight_step_<N>/` LoRA adapters
- `checkpoints/oversight_best/` symlink to best by validation F1
- HF Hub push to `<user>/admarket-oversight-qwen2.5-3b-grpo`

**Reward**: `OversightF1Rubric.score_day(...)` — `daily_f1 - 0.5 * false_positives`. False-positive penalty prevents the trivial 'flag everyone' exploit.

## 1. Install dependencies

In [ ]:
%%capture
# Colab T4 install (Apr 2026). Same recipe as train_grpo.ipynb.
#
# Hard-won lessons:
#   * Do NOT pin transformers / accelerate / datasets — Unsloth's latest needs
#     transformers>=4.47 (for `CompileConfig`); older pins silently downgrade
#     the wheel pip just installed and Unsloth then ImportErrors at import time.
#   * bitsandbytes must be >=0.45.0 — 0.44 imports the removed `triton.ops`
#     and ships no CUDA 12.8 wheel for current Colab runtimes.
#   * Install Unsloth LAST so its dependency resolver gets the final say
#     over transformers/accelerate/peft versions.
!pip install -U "trl>=0.12" "peft>=0.13"
!pip install -U "bitsandbytes>=0.45.0"
!pip install -U matplotlib
# Required by models.py / server.arena_rubrics (provides openenv.core types)
!pip install -U "openenv-core>=0.2.1"
# Unsloth pulls compatible transformers / accelerate / xformers wheels itself.
!pip install -U unsloth

## 2. Mount drive / clone repo

If you've copied the repo into the Colab runtime already, skip the clone and just `cd` into it. Otherwise pull from the team repo so the latest `oversight.py`, `violation_injector.py`, `models.py` are available.

In [ ]:
import os, sys
REPO_PATH = '/content/AdMarket-Arena'  # cloned from ritz-gupta/AdMarket-Arena
if not os.path.exists(REPO_PATH):
    !git clone -b play2 https://github.com/ritz-gupta/AdMarket-Arena.git $REPO_PATH
%cd $REPO_PATH
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

## 3. Imports + config

In [ ]:
import json
import os
import random
import textwrap
from pathlib import Path
from typing import List

import torch
from datasets import Dataset

from models import GroundTruthViolation, OversightObservation, ViolationFlag
from oversight import (
    OVERSIGHT_SYSTEM_PROMPT,
    HeuristicOversightAgent,
    _format_observation_for_prompt,
    parse_llm_flags,
    score_flags,
)
from server.arena_rubrics import OversightF1Rubric

# --- knobs ---
BASE_MODEL          = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
# Prompts (system + 30-line auction log + state) run ~2.5–3k tokens; need
# headroom for the JSON answer so set 8192 to avoid silent left-truncation
# of the system prompt (which previously killed the reward signal).
MAX_SEQ_LEN         = 8192
LORA_RANK           = 16
TRAIN_DATA_PATH     = Path('data/oversight_train_trajectories.jsonl')
VAL_FRACTION        = 0.10
OUTPUT_DIR          = Path('checkpoints')
# 64 was too tight: model never finished the JSON object → parse_llm_flags
# returned [] for every completion → reward identically 0 → std=0 → no
# advantage → no gradient. 256 covers a worst-case multi-flag JSON.
MAX_NEW_TOKENS      = 256
NUM_GENERATIONS     = 4   # GRPO group size, T4-tight
LEARNING_RATE       = 1e-5
MAX_STEPS           = 60
SAVE_EVERY_STEPS    = 10
LOG_EVERY_STEPS     = 1
HF_HUB_REPO_ID      = 'ritz-gupta/admarket-oversight-qwen2.5-3b-grpo'
SEED                = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); torch.manual_seed(SEED)
print('Config OK.')

## 4. Load model with Unsloth (4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_RANK,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
print(f'Model loaded: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable params (LoRA).')

## 5. Build dataset from oversight trajectories

Each prompt = one OversightObservation rendered into the canonical format from `oversight._format_observation_for_prompt`. The reward function below decodes the LLM's JSON output via `parse_llm_flags`, scores it against the ground truth via `OversightF1Rubric`, and returns a scalar reward to GRPO.

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

raw = load_jsonl(TRAIN_DATA_PATH)
random.Random(SEED).shuffle(raw)
split_at = max(1, int(len(raw) * (1 - VAL_FRACTION)))
raw_train, raw_val = raw[:split_at], raw[split_at:]

# Class-balance the TRAIN split only (validation stays on the natural
# 36/64 distribution so the reported F1 reflects the true environment).
#
# Why this is necessary even after fp_penalty=0.1: on a no-violation
# day, ANY flag at all yields F1=0 + FP penalty, so the empty policy
# strictly dominates trying. With 64% no-violation days in the natural
# split, the empty-policy attractor still pulls hard enough that GRPO
# collapsed to {"flags":[]} around step ~30 in the previous run
# (kl exploded to 1.23, reward_std went to 0). Downsampling to 50/50
# kills the attractor without changing the eval-time reward.
viol_train  = [r for r in raw_train if len(r['ground_truth']) > 0]
empty_train = [r for r in raw_train if len(r['ground_truth']) == 0]
n_keep = min(len(viol_train), len(empty_train))
empty_keep = random.Random(SEED).sample(empty_train, k=n_keep)
raw_train = viol_train + empty_keep
random.Random(SEED + 1).shuffle(raw_train)

viol_rate_train = sum(len(r['ground_truth']) > 0 for r in raw_train) / max(1, len(raw_train))
viol_rate_val   = sum(len(r['ground_truth']) > 0 for r in raw_val)   / max(1, len(raw_val))
print(f'Loaded {len(raw)} day-records. '
      f'train={len(raw_train)} (violation_rate={viol_rate_train:.1%}, balanced)  '
      f'val={len(raw_val)} (violation_rate={viol_rate_val:.1%}, natural)')

def build_example(row: dict) -> dict:
    obs = OversightObservation.model_validate(row['observation'])
    # 30 log rows keeps the prompt under ~3k tokens. 80 (the function
    # default) blew past MAX_SEQ_LEN=4096 and got the system prompt
    # truncated. Aggregate counts (the only thing the reward cares about)
    # are robust to log down-sampling.
    user_prompt = _format_observation_for_prompt(obs, max_log_lines=30)
    chat = tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': OVERSIGHT_SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {
        'prompt': chat,
        'ground_truth_json': json.dumps(row['ground_truth']),
    }

train_ds = Dataset.from_list([build_example(r) for r in raw_train])
val_ds   = Dataset.from_list([build_example(r) for r in raw_val])
print(train_ds[0]['prompt'][:400], '...')

## 6. Reward function (F1 with FP penalty)

In [ ]:
# Eval-time rubric — kept around so the validation cell (## 9) can still
# report the canonical F1 number. Training reward below is decoupled from
# F1 on purpose; see docstring.
_RUBRIC = OversightF1Rubric(fp_penalty=0.1)

# Per-flag training-reward weights. TP_REWARD = 1.0 means "each correct
# (advertiser_id, violation_type) flag is worth as much as one perfect
# F1 day." FP_PENALTY = 0.1 means "ten wrong flags cost the same as one
# right one." This 10:1 asymmetry favors recall over precision during
# training, which is the regime we want — the env's downstream consumer
# of these flags can apply additional precision filters.
TP_REWARD = 1.0
FP_PENALTY = 0.1

def oversight_reward_fn(prompts, completions, ground_truth_json, **_):
    """GRPO training reward — linear count-based, not F1-based.

    Three runs of F1-based rewards collapsed to the always-empty policy:
    score_flags() returns F1=1.0 when both pred and truth are empty,
    which made "{\"flags\":[]}" the dominant attractor regardless of
    fp_penalty value or KL beta. Capping the empty/empty case (run 4)
    just shifted the collapse target without preventing it, because
    F1's nonlinearity still made noisy exploration costly relative to
    the safe empty baseline.

    Linear count-based reward fixes both problems at once:
      reward = TP_REWARD * (# true positives) - FP_PENALTY * (# false positives)
    Empty completions score exactly 0 regardless of ground truth (no
    asymmetric bonus). Each correct flag contributes independently
    (linear gradient toward "find more violations").

    Eval-time F1 is unchanged — see val_f1s in cell 20. Training reward
    is a *proxy* designed to induce good F1, not equal F1. (Standard
    decoupling in RLHF: proxy reward != evaluation metric.)

    Cold-start shaping (+0.05 for parseable JSON with "flags" key) is
    kept to ensure the very first GRPO groups have non-zero std before
    the model starts producing real predictions.
    """
    rewards = []
    for completion, gt_json in zip(completions, ground_truth_json):
        text = completion if isinstance(completion, str) else completion[-1].get('content', '')
        flags: List[ViolationFlag] = parse_llm_flags(text)
        truth = [GroundTruthViolation.model_validate(t) for t in json.loads(gt_json)]
        pred_set = {(f.advertiser_id, f.violation_type) for f in flags}
        truth_set = {(t.advertiser_id, t.violation_type) for t in truth}
        tp = len(pred_set & truth_set)
        fp = len(pred_set - truth_set)
        reward = TP_REWARD * float(tp) - FP_PENALTY * float(fp)
        if '"flags"' in text and '{' in text and '}' in text:
            reward += 0.05
        rewards.append(reward)
    return rewards

# Sanity-check the new reward on a heuristic-prediction baseline.
_h = HeuristicOversightAgent()
_demo_obs = OversightObservation.model_validate(raw_val[0]['observation'])
_demo_truth = [GroundTruthViolation.model_validate(t) for t in raw_val[0]['ground_truth']]
_demo_flags = _h.flag_day(_demo_obs)
print('heuristic flags:', [(f.advertiser_id, f.violation_type) for f in _demo_flags])
print('heuristic eval-F1 reward:', _RUBRIC.score_day(_demo_flags, _demo_truth)['reward'])
# Reuse the training reward fn on the heuristic prediction so we see
# what scale the new GRPO reward will operate on.
_demo_train_reward = oversight_reward_fn(
    prompts=[''],
    completions=[json.dumps({'flags': [
        {'advertiser_id': f.advertiser_id, 'violation_type': f.violation_type, 'confidence': f.confidence}
        for f in _demo_flags
    ]})],
    ground_truth_json=[json.dumps(raw_val[0]['ground_truth'])],
)[0]
print('heuristic training reward (linear):', round(_demo_train_reward, 3))

## 7. SFT cold-start (oracle imitation)

Four runs of pure GRPO collapsed to the empty-flag policy: random exploration of the LLM's output space produces false positives faster than true positives, so the model never gets enough positive reward signal to escape the "safe empty" equilibrium. The diagnosis is structural — base Qwen2.5-3B has no idea what a frequency-cap or shill-bidding violation *looks like* in an auction log, so its initial flag distribution is essentially noise.

**Fix**: standard RLHF recipe. Supervised fine-tuning on `(observation → ground-truth flags)` pairs first, then GRPO refines on top. After ~4 epochs the LoRA adapter knows the JSON output format AND knows what violations look like in the data; the post-SFT model produces TPs more often than FPs, so GRPO's gradient finally points toward "do more of what's working" instead of "play it safe."

**SFT details**:
- 80 steps × effective batch 8 ≈ 4 epochs over the 136 balanced training rows
- Higher LR (2e-4) than GRPO since LoRA SFT is stable at higher learning rates
- Same model object — LoRA adapter accumulates SFT weight updates that GRPO will then refine
- Confidence in the supervised JSON is a constant 0.95 (the reward fn ignores confidence anyway, matching only on `(advertiser_id, violation_type)`)

Expected runtime on T4: ~5–8 minutes. After this cell, GRPO in cell 16 starts from a model that already produces ~70%-correct flag predictions instead of pure noise.

In [ ]:
import torch
from trl import SFTConfig, SFTTrainer


class AssistantOnlyCollator:
    """Mask everything before the assistant turn so SFT loss is computed
    only on the JSON output, not on the (essentially constant) prompt.

    Equivalent to TRL's `DataCollatorForCompletionOnlyLM` but written
    inline so we don't depend on a specific TRL minor version (the class
    moved between trl, trl.trainer, and trl.trainer.utils across
    versions; some recent releases dropped it entirely in favor of
    `assistant_only_loss=True` which requires `{% generation %}` tags
    in the chat template that Qwen2.5 doesn't have).
    """
    def __init__(self, tokenizer, response_template: str = '<|im_start|>assistant\n'):
        self.tokenizer = tokenizer
        self.template_ids = tokenizer(response_template, add_special_tokens=False)['input_ids']
        self.pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    def __call__(self, examples):
        # SFTTrainer pre-tokenizes the dataset internally, so by the
        # time we run inside training, examples have 'input_ids' (and
        # 'attention_mask'). The sanity-check below calls us with raw
        # 'text' rows from the original Dataset. Handle both.
        if 'input_ids' in examples[0]:
            max_len = max(len(ex['input_ids']) for ex in examples)
            input_ids = torch.full((len(examples), max_len), self.pad_id, dtype=torch.long)
            attention_mask = torch.zeros((len(examples), max_len), dtype=torch.long)
            for i, ex in enumerate(examples):
                ids = ex['input_ids']
                input_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
                mask = ex.get('attention_mask', [1] * len(ids))
                attention_mask[i, :len(mask)] = torch.tensor(mask, dtype=torch.long)
        else:
            texts = [ex['text'] for ex in examples]
            enc = self.tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
            input_ids = enc['input_ids']
            attention_mask = enc['attention_mask']

        labels = input_ids.clone()
        n = len(self.template_ids)
        for i in range(labels.size(0)):
            ids = input_ids[i].tolist()
            split = -1
            for j in range(len(ids) - n + 1):
                if ids[j:j+n] == self.template_ids:
                    split = j + n
                    break
            if split > 0:
                labels[i, :split] = -100
            else:
                # Marker not found — mask everything (safe-fail; this
                # example will contribute zero gradient).
                labels[i, :] = -100
            labels[i, attention_mask[i] == 0] = -100
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }

def _format_target_json(ground_truth: list[dict]) -> str:
    """Canonical JSON answer for SFT supervision.

    Format matches OVERSIGHT_SYSTEM_PROMPT's spec: a single JSON object
    with a "flags" array of {advertiser_id, violation_type, confidence}.
    Confidence is fixed at 0.95 — the reward function only matches on
    (advertiser_id, violation_type), so confidence is just a placeholder
    that GRPO will later learn to calibrate on its own.
    """
    flags = [
        {
            'advertiser_id': int(t['advertiser_id']),
            'violation_type': t['violation_type'],
            'confidence': 0.95,
        }
        for t in ground_truth
    ]
    return json.dumps({'flags': flags}, separators=(', ', ': '))

def _build_sft_row(row: dict) -> dict:
    """Pre-tokenize each example into a single chat-formatted text
    string. Unsloth's patched SFTTrainer requires either a `text` field
    or an explicit formatting_func — pre-tokenizing avoids the latter
    and is also marginally faster (one apply_chat_template call per
    example at dataset-build time, not per training step).
    """
    obs = OversightObservation.model_validate(row['observation'])
    user_prompt = _format_observation_for_prompt(obs, max_log_lines=30)
    target = _format_target_json(row['ground_truth'])
    chat_text = tokenizer.apply_chat_template(
        [
            {'role': 'system',    'content': OVERSIGHT_SYSTEM_PROMPT},
            {'role': 'user',      'content': user_prompt},
            {'role': 'assistant', 'content': target},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': chat_text}

sft_train_ds = Dataset.from_list([_build_sft_row(r) for r in raw_train])
print(f'SFT dataset: {len(sft_train_ds)} examples')
# Show only the assistant tail so we can verify the supervised target
# was assembled correctly without dumping the whole 2.5k-token prompt.
_sample_text = sft_train_ds[0]['text']
_assistant_marker = '<|im_start|>assistant'
_tail_idx = _sample_text.rfind(_assistant_marker)
print('Sample assistant turn:', _sample_text[_tail_idx:_tail_idx + 300] if _tail_idx >= 0 else _sample_text[-300:])

sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR / 'oversight_sft'),
    learning_rate=1e-4,             # was 2e-4; lower for stability over longer run
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch 8 → ~17 batches/epoch
    max_steps=200,                  # was 80; ~12 epochs (loss-only-on-assistant fix
                                    # below makes the per-step gradient much more
                                    # informative, but we still need enough steps
                                    # to actually learn the discrimination)
    save_steps=200,
    logging_steps=10,
    bf16=False,
    fp16=True,
    optim='adamw_8bit',
    report_to=[],
    seed=SEED,
    max_seq_length=MAX_SEQ_LEN,
    warmup_ratio=0.1,
)

# Mask out everything before the assistant turn so the loss is computed
# only on the JSON output, not on the (essentially constant) 2.5k-token
# prompt. Without this, ~98% of the loss signal comes from predicting
# the prompt back to itself — easy and uninformative — and the model
# converges to the most-common assistant turn (empty) regardless of
# input. This was the root cause of the train-F1=0.400 / val-F1=0.619
# diagnostic in the previous run (both numbers consistent with "always
# emit empty, with occasional noise on memorized train examples").
sft_collator = AssistantOnlyCollator(tokenizer)

# Sanity-check: tokenize one example and confirm our masking finds the
# assistant marker. If the next print shows masked_tokens=full_length
# (or 0), the response_template doesn't match the tokenized output and
# we'd silently train on nothing.
_probe = sft_collator([sft_train_ds[0]])
_total_tokens = (_probe['attention_mask'] == 1).sum().item()
_unmasked     = (_probe['labels'] != -100).sum().item()
print(f'Mask sanity-check: {_unmasked}/{_total_tokens} tokens contribute to loss '
      f'({_unmasked / _total_tokens:.1%}).')
assert 0 < _unmasked < _total_tokens, (
    'Mask is degenerate. If 0: response_template does not match the chat '
    'template output. If full: nothing was masked. Inspect tokenizer.decode'
    '(_probe["input_ids"][0]) to see actual chat formatting.'
)

sft_trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=sft_train_ds,
    data_collator=sft_collator,
)
sft_trainer.train()
print('SFT cold-start complete (loss-on-assistant-only, 200 steps).')
print('Diagnostic: re-run the train-F1 + val-F1 cells to confirm SFT learned '
      'before launching GRPO. Expected: train-F1 > 0.7, val-F1 > 0.5.')

## 8. GRPO trainer + run

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=str(OUTPUT_DIR / 'oversight_run'),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_SEQ_LEN - MAX_NEW_TOKENS,
    max_completion_length=MAX_NEW_TOKENS,
    max_steps=MAX_STEPS,
    save_steps=SAVE_EVERY_STEPS,
    logging_steps=LOG_EVERY_STEPS,
    bf16=False,
    fp16=True,
    optim='adamw_8bit',
    # Reverted to TRL default (0.04). With the linear count-based reward
    # in cell 12 there is no longer a strong empty-policy attractor for
    # KL to fight against, so the policy needs more mobility — not less
    # — to discover which (advertiser, violation_type) pairs are worth
    # flagging. If kl shoots past 0.4 by step ~30 again, raise back to
    # 0.07–0.10; if reward is climbing but slowly, leave alone.
    # beta=0.04,  # TRL default; not setting explicitly
    report_to=[],
    remove_unused_columns=False,
    seed=SEED,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[oversight_reward_fn],
    args=config,
    train_dataset=train_ds,
)
trainer.train()

## 8.5 Training curves + run-log dump

TRL stashes everything it logs (loss, per-reward-fn mean/std, completion lengths, kl, clipped_ratio) in `trainer.state.log_history`. We do two things with it:

1. **3-panel plot** of the metrics that actually matter for diagnosing GRPO health: training loss, mean reward, and reward std. The `reward_std` curve is the early-warning signal — if it collapses to 0 for many consecutive steps, GRPO has degenerated (all generations in a group agree, advantage = 0, no learning) and the run should be killed.
2. **CSV dump** of the full log_history to `logs/oversight_<timestamp>.csv` so the run is re-plottable later without having to re-train.

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt

LOGS_DIR = Path('logs'); LOGS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = Path('assets/plots'); PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Newer TRL log_history rows have either a 'loss' key (training step) or
# only eval/runtime keys (epoch summaries). Keep only the per-step rows.
hist = pd.DataFrame(trainer.state.log_history)
hist = hist[hist['loss'].notna()].reset_index(drop=True)

ts = time.strftime('%Y%m%d_%H%M%S')
csv_path = LOGS_DIR / f'oversight_{ts}.csv'
hist.to_csv(csv_path, index=False)
print(f'Wrote {len(hist)} step rows to {csv_path}')

reward_mean_col = 'rewards/oversight_reward_fn/mean'
reward_std_col  = 'rewards/oversight_reward_fn/std'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(hist['step'], hist['loss'], color='tab:blue')
axes[0].set_title('Training loss')
axes[0].set_xlabel('step'); axes[0].set_ylabel('loss')
axes[0].axhline(0, color='gray', lw=0.5, ls='--')

axes[1].plot(hist['step'], hist[reward_mean_col], color='tab:green')
axes[1].set_title('Reward (group mean)')
axes[1].set_xlabel('step'); axes[1].set_ylabel('reward')
axes[1].axhline(0, color='gray', lw=0.5, ls='--')

axes[2].plot(hist['step'], hist[reward_std_col], color='tab:orange')
axes[2].set_title('Reward std (gradient health — keep > 0)')
axes[2].set_xlabel('step'); axes[2].set_ylabel('std')
axes[2].axhline(0, color='gray', lw=0.5, ls='--')

plt.tight_layout()
plot_path = PLOTS_DIR / f'oversight_training_curves_{ts}.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved plot to {plot_path}')

zero_std_steps = int((hist[reward_std_col] == 0).sum())
print(f'\nDiagnostic: {zero_std_steps}/{len(hist)} steps had reward_std == 0  '
      f'(those steps contributed no gradient).')

## 9. Validation F1 + best-checkpoint selection

In [ ]:
FastLanguageModel.for_inference(model)

def _generate(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

val_f1s = []
for example in val_ds:
    text = _generate(example['prompt'])
    flags = parse_llm_flags(text)
    truth = [GroundTruthViolation.model_validate(t) for t in json.loads(example['ground_truth_json'])]
    val_f1s.append(score_flags(flags, truth).f1)
import statistics as _stats
val_f1_mean = _stats.mean(val_f1s) if val_f1s else 0.0
print(f'Validation mean F1: {val_f1_mean:.3f}  (n={len(val_f1s)})')

# Save best checkpoint (overwrite the symlink to the latest if best).
BEST_PATH = OUTPUT_DIR / 'oversight_best'
model.save_pretrained(str(BEST_PATH))
tokenizer.save_pretrained(str(BEST_PATH))
print(f'Saved best checkpoint to {BEST_PATH}')

## 10. Push to HuggingFace Hub

In [ ]:
# from huggingface_hub import HfApi, login
# login()  # paste token

# model.push_to_hub(HF_HUB_REPO_ID, private=False)
# tokenizer.push_to_hub(HF_HUB_REPO_ID, private=False)

# model_card = textwrap.dedent(f"""\
# ---
# language: en
# license: apache-2.0
# library_name: peft
# tags:
# - grpo
# - unsloth
# - oversight
# - ad-tech
# base_model: {BASE_MODEL}
# ---

# # AdMarket Arena — Trained Oversight Agent

# GRPO-trained LoRA adapter for the AdMarket Arena env's OversightAgent role. Detects three violation types (frequency_cap, budget_overspend, shill_bidding) from auction logs.

# **Training**: {MAX_STEPS} GRPO steps, num_generations={NUM_GENERATIONS}, lr={LEARNING_RATE}, on Colab T4. Reward function: `OversightF1Rubric.score_day` (daily F1 minus 0.5 per false positive).

# **Validation F1**: {val_f1_mean:.3f}

# **Eval results**: see `oversight_eval_results.json` in the env repo, generated by `scripts/oversight_eval.py`.
# """)
# Path('MODEL_CARD.md').write_text(model_card)
# HfApi().upload_file(path_or_fileobj='MODEL_CARD.md', path_in_repo='README.md', repo_id=HF_HUB_REPO_ID, repo_type='model')
# print('Pushed to', HF_HUB_REPO_ID)